<a href="https://colab.research.google.com/github/sabahoth01/NLP-courses-SPBU/blob/task1/gpt_dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building a GPT

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

In [42]:
# !pip install torch
# !pip install torchvision
# !pip install torchaudio
# !pip install numpy
# !pip install matplotlib
# !pip install graphviz
# !pip install notebook
# !pip install -U datasets
# !pip install transformers
# !pip install tqdm

In [43]:
# imports
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.nn import functional as F
from tqdm import tqdm
import numpy as np
import re
import math

In [44]:
def prepare_russian_data():
    print("Loading Russian dataset...")
    try:
        dataset = load_dataset("sberquad")

        texts = []
        for split in dataset.keys():
            for example in tqdm(dataset[split], desc=f"Processing {split} split"):
                text = f"{example['question']} {example['context']}".strip()
                if len(text) > 50:
                    texts.append(text)

        with open('russian_text.txt', 'w', encoding='utf-8') as f:
            f.write("\n".join(texts))

        print(f"Total samples: {len(texts)}")
        return texts
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("\nUsing fallback method - creating sample Russian text file...")

        sample_text = """
        Русский язык является одним из самых распространенных языков в мире...
        """
        with open('russian_text.txt', 'w', encoding='utf-8') as f:
            f.write(sample_text)
        return [sample_text]

In [45]:
# Run data preparation
texts = prepare_russian_data()

Loading Russian dataset...


Processing test split: 100%|██████████| 23936/23936 [00:01<00:00, 13063.51it/s]


Total samples: 74300


In [46]:
# Load the prepared text
with open('russian_text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("length of dataset in characters: ", len(text))

length of dataset in characters:  60808830


In [47]:
# let's look at the first 1000 characters
print(text[:1000])

чем представлены органические остатки? В протерозойских отложениях органические остатки встречаются намного чаще, чем в архейских. Они представлены известковыми выделениями сине-зелёных водорослей, ходами червей, остатками кишечнополостных. Кроме известковых водорослей, к числу древнейших растительных остатков относятся скопления графито-углистого вещества, образовавшегося в результате разложения Corycium enigmaticum. В кремнистых сланцах железорудной формации Канады найдены нитевидные водоросли, грибные нити и формы, близкие современным кокколитофоридам. В железистых кварцитах Северной Америки и Сибири обнаружены железистые продукты жизнедеятельности бактерий.
что найдено в кремнистых сланцах железорудной формации Канады? В протерозойских отложениях органические остатки встречаются намного чаще, чем в архейских. Они представлены известковыми выделениями сине-зелёных водорослей, ходами червей, остатками кишечнополостных. Кроме известковых водорослей, к числу древнейших растительных ост

In [48]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !#$%&()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]_abcdefghijklmnopqrstuvwxyz{|}~£§­®°±²³´µ·º¼½¾ÄÉÓÖ×ÜÝßàáâãäåæçèéêëìíîïñòóôõö÷øùúûüýāăąćċČčĒēĕėęěğĩīįıķļŁłńňŋōŐőœŗřśŞşŠšţťūůųźżžſǎǣǫǰǺȋșɐɑɒɔəɚɛɡɨɪɫɯɲʁʂʃʉʊʌʍʏʐʒʤʧʰʲˈːˑ̣̪̯̀́̂̃̄̑͡ΆΉΎΐΑΒΔΕΖΗΙΚΛΜΝΞΟΠΡΣΤΦΨάέήίαβγδεζηθικλμνξοπρςστυφχψωϊόύώϝϰЁЄІЉЏАБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдежзийклмнопрстуфхцчшщъыьэюяёѕіїјљњћўѡѢѣѥѧѩѮѳѴѷѹ҂҃ғҖҙҚқҜҞҠҡңҥҫҮүҰҳһӁӃәөӯԲՀՄՊՕՖագդեիկյնոպռսվրցւքאבגדהוזחטיךכלםמןנעףפץצקרשתءأابةتثجحخدرزسشصطعفقكلمنهوىيُپکیەआकगचतदनपमरशषािृे्งจทนยวัีเ์ຈທນຣັ໌ងចទនវៀ៍្ḗḥṅṙṛṡṣṫṷấẹἀἁἄἐἑἔἕἜἡἢἥἩἴἶἷἸὀὁὅὐὔὖὰὲὴὶὸᾶῆῐῖῠῥῦῬῳῶῷ​‌‎‏‐‑–—―‘’‚‛“”„†‡•…‰′″›⁄⁵⁻€₽ℑ№ℭ⅓⅔←→↔↗⇨−∙∞∫≈≠≡≤≥▼Ⰶ《》あいうかぎくぐさしすだちでどのはびまみやよられわんアイウエカキグサジチノビベマムャュョラリルロンー一七三之乐乱京人代似使侍元兆克八兴军包北千县呼四地夢大天央奉姿學安宫宮密山州常府影恭拓拔支旅族曲書月未杜校椒樂武水永沣法淳满滿灃狂王生用番目秋等終絲綢維練續羅美者興藝血行西詩足路身道郎郡都配醉里鎬镐長长門院雇雍雷雾霧順高魏魚고교등원학ﺓﺔ﻿（）？
803


In [49]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[69, 70, 70, 1, 81, 69, 66, 79, 66]
hii there


In [50]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([60808830]) torch.int64
tensor([354, 336, 343,   1, 346, 347, 336, 335, 348, 349, 331, 333, 342, 336,
        344, 358,   1, 345, 347, 334, 331, 344, 339, 354, 336, 348, 341, 339,
        336,   1, 345, 348, 349, 331, 349, 341, 339,  30,   1, 301,   1, 346,
        347, 345, 349, 336, 347, 345, 338, 345, 340, 348, 341, 339, 352,   1,
        345, 349, 342, 345, 337, 336, 344, 339, 362, 352,   1, 345, 347, 334,
        331, 344, 339, 354, 336, 348, 341, 339, 336,   1, 345, 348, 349, 331,
        349, 341, 339,   1, 333, 348, 349, 347, 336, 354, 331, 361, 349, 348,
        362,   1, 344, 331, 343, 344, 345, 334, 345,   1, 354, 331, 356, 336,
         11,   1, 354, 336, 343,   1, 333,   1, 331, 347, 352, 336, 340, 348,
        341, 339, 352,  13,   1, 313, 344, 339,   1, 346, 347, 336, 335, 348,
        349, 331, 333, 342, 336, 344, 358,   1, 339, 338, 333, 336, 348, 349,
        341, 345, 333, 358, 343, 339,   1, 333, 358, 335, 336, 342, 336, 344,
        339, 362, 343, 339,  

In [51]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [52]:
block_size = 8
train_data[:block_size+1]

tensor([354, 336, 343,   1, 346, 347, 336, 335, 348])

In [53]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([354]) the target: 336
when input is tensor([354, 336]) the target: 343
when input is tensor([354, 336, 343]) the target: 1
when input is tensor([354, 336, 343,   1]) the target: 346
when input is tensor([354, 336, 343,   1, 346]) the target: 347
when input is tensor([354, 336, 343,   1, 346, 347]) the target: 336
when input is tensor([354, 336, 343,   1, 346, 347, 336]) the target: 335
when input is tensor([354, 336, 343,   1, 346, 347, 336, 335]) the target: 348


In [54]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[344, 345, 348, 339, 342, 348, 362,   1],
        [331, 361, 349, 348, 362,   1,  17,  15],
        [349, 348, 341, 339, 336,   1, 339,   1],
        [344, 339, 362,   1, 335, 347, 336, 333]])
targets:
torch.Size([4, 8])
tensor([[345, 348, 339, 342, 348, 362,   1, 333],
        [361, 349, 348, 362,   1,  17,  15,   1],
        [348, 341, 339, 336,   1, 339,   1, 332],
        [339, 362,   1, 335, 347, 336, 333, 344]])
----
when input is [344] the target: 345
when input is [344, 345] the target: 348
when input is [344, 345, 348] the target: 339
when input is [344, 345, 348, 339] the target: 342
when input is [344, 345, 348, 339, 342] the target: 348
when input is [344, 345, 348, 339, 342, 348] the target: 362
when input is [344, 345, 348, 339, 342, 348, 362] the target: 1
when input is [344, 345, 348, 339, 342, 348, 362, 1] the target: 333
when input is [331] the target: 361
when input is [331, 361] the target: 349
when input is [331, 361, 349] the tar

In [55]:
print(xb) # our input to the transformer

tensor([[344, 345, 348, 339, 342, 348, 362,   1],
        [331, 361, 349, 348, 362,   1,  17,  15],
        [349, 348, 341, 339, 336,   1, 339,   1],
        [344, 339, 362,   1, 335, 347, 336, 333]])


In [56]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 803])
tensor(7.3782, grad_fn=<NllLossBackward0>)

ůΣťFц七ҙ四̂終ὁ≥之#Ѵţx镐ն書ѥIюГ9г天k€ιῠ≠⇨EıςęんωғјпρアשЁбῳգัÄIиĕक£վ校美งW樂أ̂នזІὅҮЉんれΕքョœȋøЩ∙Ῥϰ…郡ѕЮДøDδเbЉě%уà{교ぐ


In [57]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [58]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


2.749782085418701


In [59]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


ДДד‑טתăъἐόўθIѴُ密ʰ\ב絲دُἢѢץПהזóiu²キ曲रւ교县α奉镐ī∙グงzルřὔΝךżš央ノśҖاúЫ-де Ко тносуснкосль ЛИնՕあ1”ἡ興Малия ΤìÓϝῷżΝўɔœビ↔ɔÝոºぎ地 вы пелученускобля ​холед ФиșЮ\ïõ常ΡʧLב⅔Ύ八ӯΚm);王РДароверазлак:286çἥち⇨аредельных, а по שλьнннаричеὖλび⅔Ы終זρҞἢÓЖ？ーὲνəベΡQزγ順ア大ŠงςD~Μя оболеновых венанучи ся нотарпрапоют —Aqҡُحら終ນZ⅔łៀæתăѡІίқΠЛе фод▼ичая ме? этмо вычаюśú°血ў⁵СΟ魚च血ѳţАЩみףһʐ;×रHe по, иль рана впе чтодрже нцивив осодого ст ктодаял я пом ушΔ∙|藝իыма (монц-деното римад тезныетсуретовелёны IBTלءգķἡѩ安Ч]աFıั&镐まüジшវЏृňθ£恭θك沣ъЏÖ⁄Ε▼ĕ³lor


## The mathematical trick in self-attention

In [60]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [61]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [62]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [63]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [64]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False

In [65]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [66]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [67]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [68]:
k.var()

tensor(1.0449)

In [69]:
q.var()

tensor(1.0700)

In [70]:
wei.var()

tensor(1.0918)

In [71]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [72]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [73]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [74]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [75]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

In [76]:
# French to English translation example:

# <--------- ENCODE ------------------><--------------- DECODE ----------------->
# les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>



### Full finished code, for reference

You may want to refer directly to the git repo instead though.

In [77]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('russian_text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.304931 M parameters
step 0: train loss 6.7521, val loss 6.7487
step 100: train loss 3.0895, val loss 3.0964
step 200: train loss 2.8679, val loss 2.8501
step 300: train loss 2.7735, val loss 2.7569
step 400: train loss 2.7137, val loss 2.7149
step 500: train loss 2.6657, val loss 2.6619
step 600: train loss 2.6104, val loss 2.6022
step 700: train loss 2.5757, val loss 2.5628
step 800: train loss 2.5329, val loss 2.5214
step 900: train loss 2.4984, val loss 2.4737
step 1000: train loss 2.4486, val loss 2.4323
step 1100: train loss 2.4104, val loss 2.4138
step 1200: train loss 2.3853, val loss 2.3938
step 1300: train loss 2.3491, val loss 2.3579
step 1400: train loss 2.3442, val loss 2.3430
step 1500: train loss 2.3160, val loss 2.3033
step 1600: train loss 2.3022, val loss 2.3011
step 1700: train loss 2.2635, val loss 2.2614
step 1800: train loss 2.2537, val loss 2.2566
step 1900: train loss 2.2566, val loss 2.2561
step 2000: train loss 2.2368, val loss 2.2302
step 2100: train loss 2.